In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("DSC100_Lab4.ipynb")

# 🕵️ LAB 4: Food Safety (Continued)
## Cleaning Data and EDA

## 👥 Collaboration Policy

Data science is a collaborative activity. While you may talk with others about the homework, we ask that you **write your solutions individually**. If you do discuss the assignments with others, please **include their names** below.

**Collaborators**: *list collaborators here*


## 📝 This Assignment

This lab will contain two parts. In the first part, you will warm up by doing some pandas data cleaning and EDA with our known datasets regarding elections and names. In the second part, we will continue our exploration of restaurant food safety scores for restaurants in San Francisco. The main goal for this assignment is to focus more on the analysis of the dataset, building on the data cleaning we have done earlier in Lab 3. 


After this homework, you should be comfortable with:
* Reading `pandas` documentation and using `pandas` methods,
* Working with data at different levels of granularity,
* Using `groupby` with different aggregation functions, and
* Chaining different `pandas` functions and methods to find answers to exploratory questions.


## Score Breakdown 
Question | Manual | Points
--- | --- | ---
1a | no | 2
1b | no | 2
1c | no | 2
2 | no |  3
3 | no | 3
4 | no | 3
5a | no | 2
5b | no | 3
5c | no | 3
2a | no | 2
6b | no | 3
6c | no | 3
7a | yes | 4
7b | yes | 4
8a | no | 2
8b | no | 2
8c | no | 3
8d | no | 3
8e | yes | 1
Total | 3 | 50


## 🏎️ Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question. 

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

**Important note: The local autograder tests will not be comprehensive. You can pass the automated tests in your notebook but still fail tests in the autograder.** Please be sure to check your results carefully.

Finally, unless we state otherwise, **do not use for loops or list comprehensions**. The majority of this assignment can be done using built-in commands in `pandas` and `NumPy`. Our autograder isn't smart enough to check, but you're depriving yourself of key learning objectives if you write loops / comprehensions, and you also won't be ready for the midterm.

### 🐛 Debugging Guide
If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Jupyter notebooks / Datahub, Gradescope, and common `pandas` errors.

In [ ]:
import numpy as np
import pandas as pd

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Part 1

---
[`pandas`](https://pandas.pydata.org/) is one of the most widely used `Python` libraries in data science. In this lab, you will review commonly used data-wrangling operations/tools in `pandas`. We continue the content from the previous lab and aim to give you familiarity with:

* Aggregating the data (using `.groupby`),
* Filtering the data (using boolean arrays and `groupby.filter`),
* Pivoting (using `.pivot_table`).

In this lab, you are going to use several `pandas` methods. Reminder from the lecture that you may press `shift+tab` on method parameters to see the documentation for that method. For example, if you were using the `drop` method in `pandas`, you could press `shift+tab` to see what `drop` is expecting.

`pandas` is very similar to the `datascience` library that you saw in Data 8. 

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline

### **REVIEW:** `Groupby` and `Groupby` Shorthand

Let's now turn to use `groupby` from lectures 3 and 4.

### Elections

Let's start by reading in the election dataset from the `pandas` lectures.

In [ ]:
# Run this cell to load data from CSV file; no further action is needed.
elections = pd.read_csv("data/elections.csv")
elections.head(5)

As we saw before, we can `groupby` a specific column, e.g., `"Party"` and can print out the resulting sub-DataFrames. The output below can help you get an understanding of what `groupby` is actually doing.

An example is given below for elections since 1980

In [ ]:
# Run this cell to print sub-DataFrames of a groupby object; no further action is needed.
for n, g in elections[elections["Year"] >= 1980].groupby("Party"):
    print(f"Name: {n}") # By the way, this is an "f string", a relatively new and great feature of Python
    display(g)

Recall that once we've formed groups, we can aggregate each sub-DataFrame (a.k.a. group) into a single row using an aggregation function. For example, if we use `.agg('mean')` on the groups above, we get back a single `DataFrame` where each group has been replaced by a single row. In each column for that aggregate row, the value that appears is the average of all values in that group.

For columns that are non-numeric, e.g., `"Result"`, the `pandas` version we're using (version 2.0.2) will error because we cannot compute the mean of the `Result` column. To remedy this, we add a `numeric_only=True` argument to our function calls so that we only calculate the `mean` for columns that contain numeric values. Alternatively, we can manually select only the numerical columns before calling the `agg` function so the aggregation is only applied to numerical columns.

In [ ]:
elections_after_1980 = elections[elections["Year"] >= 1980]

elections_after_1980.groupby("Party").agg('mean', numeric_only=True)

# alternatively, we can manually select only the numerical columns before calling `agg`
# elections_after_1980.groupby("Party")[['Year', 'Popular vote', '%']].agg('mean')

Equivalently we can use one of the shorthand aggregation functions, e.g. `.mean()`: 

In [ ]:
elections_after_1980.groupby("Party").mean(numeric_only=True)

Note that the index of the `DataFrame` returned by a `groupby.agg` call is no longer a set of numeric indices from $0$ to $N-1$. Instead, we see that the index for the example above is now the `Party`. If we want to restore our `DataFrame` so that `Party` is a column rather than the index, we can use `reset_index`.

In [ ]:
elections_after_1980.groupby("Party").mean(numeric_only=True).reset_index()

**IMPORTANT NOTE:** Notice that the code above consists of chained method calls. This sort of code is very common in `pandas` programming and in data science in general. Such chained method calls can sometimes go many layers deep, so you might consider adding newlines between lines of code for clarity. For example, we could instead write the code above as:

In [ ]:
# pandas method chaining
(
elections.query("Year >= 1980").groupby("Party") 
                               .mean(numeric_only=True)  ## Computes the mean values by party
                               .reset_index()            ## Resets to a numerical index
)

Note that we have surrounded the entire call by a big set of parentheses so that `Python` doesn't complain about the indentation. An alternative is to use the \ symbol to indicate to `Python` that your code continues on to the next line!

In [ ]:
# pandas method chaining (alternative)
elections[elections["Year"] >= 1980].groupby("Party") \
                               .mean(numeric_only=True) \
                               .reset_index()     

**IMPORTANT NOTE:** You should NEVER solve problems like the one above using loops or list comprehensions. This is slow and also misses the entire point of this part of Data 100. 

Before we continue, we'll print out the election dataset again for your convenience. 

In [ ]:
elections.head(5)

<br>

---

### Question 1a
Using `groupby.agg` or one of the shorthand methods (`groupby.min`, `groupby.first`, etc.), create a `Series` object `best_result_percentage_only` that returns a `Series` showing the entire best result for every party, sorted in decreasing order. Your `Series` should include only parties that have earned at least 10% of the vote in some election. Your result should look like this:

<code>
Party
Democratic               61.344703
Republican               60.907806
Democratic-Republican    57.210122
National Union           54.951512
Whig                     53.051213
Liberal Republican       44.071406
National Republican      43.796073
Northern Democratic      29.522311
Progressive              27.457433
American                 21.554001
Independent              18.956298
Southern Democratic      18.138998
American Independent     13.571218
Constitutional Union     12.639283
Free Soil                10.138474
Name: %, dtype: float64
</code>
<br/>

A list of named `groupby.agg` shorthand methods are [here](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#aggregation) (you'll have to scroll down about one page).


In [ ]:
...
best_result_percentage_only

In [ ]:
grader.check("q1a")

<br>

---

### Question 1b  
Repeat Question 1a. However, this time, your result should be a `DataFrame` showing all available information (all columns) rather than only the percentage as a `Series`.

This question is trickier than Question 1a. Make sure to check the `pandas` lecture slides if you're stuck! It's very easy to make a subtle mistake that shows Woodrow Wilson and William Taft both winning the 2020 election.

For example, the first 3 rows of your table should be:

|Party | Year | Candidate      | Popular Vote | Result | %         |
|------|------|----------------|--------------|--------|-----------|
|**Democratic**  | 1964 | Lyndon Johnson | 43127041      | win   | 61.344703 |
|**Republican**  | 1972 | Richard Nixon | 47168710      | win   | 60.907806 |
|**Democratic-Republican**  | 1824 | Andrew Jackson | 151271      | loss   | 57.210122 |

Note that the index is `Party`. In other words, don't use `reset_index`.


In [ ]:
...
best_result

In [ ]:
grader.check("q1b")

### **REVIEW:** `DataFrameGroupBy.filter`

Our `DataFrame` contains a number of parties that have never had a successful presidential run. For example, the 2020 elections included candidates from the Libertarian and Green parties, neither of which have elected a president.

In [ ]:
# Run this cell to print the last four rows; no further action is needed.
elections.tail(4)

Suppose we were conducting an analysis trying to focus our attention on parties that had elected a president. 

The most natural approach is to use `groupby.filter` [(docs)](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html). This is an incredibly powerful but subtle tool for filtering data.

The code below accomplishes the task at hand. It does this by creating a function that returns `True` if and only if a sub-`DataFrame` (a.k.a. group) contains at least one winner. This function, in turn, uses the `pandas` function `any` [(docs)](https://pandas.pydata.org/docs/reference/api/pandas.Series.any.html).

In [ ]:
# Run this cell to keep only the rows of parties that have 
# elected a president; no further action is needed.
def at_least_one_candidate_in_the_frame_has_won(frame):
    """Returns df with rows only kept for parties that have
    won at least one election
    """
    return (frame["Result"] == 'win').any()

winners_only = (
    elections
        .groupby("Party")
        .filter(at_least_one_candidate_in_the_frame_has_won)
)
winners_only.tail(5)

Alternately, we could have used a `lambda` function instead of explicitly defining a named function using `def`. 

In [ ]:
# Run this cell to keep only the rows of parties that have 
# elected a president; no further action is needed.
winners_only = (
    elections
        .groupby("Party")
        .filter(lambda x : (x["Result"] == "win").any())
)
winners_only.tail(5)

<br>

---

### Question 1c

Using `filter`, create a `DataFrame` object `major_party_results_since_1988` that includes all election results starting after 1988 (exclusive) but only shows a row if the Party it belongs to has earned at least 1% of the popular vote in ANY election since 1988.

For example, despite having candidates in 2004, 2008, and 2016, no Constitution party candidates should be included since this party has not earned 1% of the vote in any election since 1988. However, you should include the Reform candidate from 2000 (Pat Buchanan) despite only having 0.43% of the vote, because in 1996, the Reform candidate Ross Perot exceeded 1% of the vote.

For example, the first three rows of the table you generate should look like:

|     |   Year | Candidate         | Party       |   Popular vote | Result   |         % |
|----:|-------:|:------------------|:------------|---------------:|:---------|----------:|
| 139 |   1992 | Andre Marrou      | Libertarian |       290087   | loss     | 0.278516  |
| 140 |   1992 | Bill Clinton      | Democratic  |       44909806 | win      | 43.118485 |
| 142 |   1992 | George H. W. Bush | Republican  |       39104550 | loss     |  37.544784|

*Hint*: The following questions might help you construct your solution. One of the lines should be identical to the `filter` examples shown above.

1) How can we **only** keep rows in the data starting after 1988 (exclusive)?
2) What column should we `groupby` to filter out parties that have earned at least 1% of the popular vote in ANY election since 1988?
3) How can we write an aggregation function that takes a sub-DataFrame and returns whether at least 1% of the vote has been earned in that sub-DataFrame? This may give you a hint about the second question!


In [ ]:
...
major_party_results_since_1988.head()

In [ ]:
grader.check("q1c")

### **REVIEW:** `str`

`pandas` provides special purpose functions for working with specific common data types such as strings and dates, which you will learn about in more detail in Lecture 6. For example, the code below provides the length of every Candidate's name from our `elections` dataset. 

In [ ]:
elections["Candidate"].str.len()

<br>

---

### Question 2

Using `.str.split`, create a new `DataFrame` called `elections_with_first_name` with a new column `First Name` that is equal to the Candidate's first name.

See the `pandas` `str` [documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html) for documentation on using `str.split`.

In [ ]:
elections_with_first_name = elections.copy()
...
elections_with_first_name

In [ ]:
grader.check("q2")

<br>

---

## Babynames
Remember the `babynames` dataset from Lab02A? Let's load it in again and explore the data with our newly covered functions! Like last time, we'll only load in data from California. 

Run the following cell: 

In [ ]:
file_path = 'data/namesbystate_ca.txt.gz'
column_labels = ['State', 'Sex', 'Year', 'Name', 'Count']

babynames = pd.read_csv(file_path, names=column_labels)

babynames.head()

The code below creates a table with the frequency of all names from 2022. 

In [ ]:
# Run this cell to create a table with the frequency 
# of all names from 2022; no further action is needed.
babynames_2022 = (
    babynames[babynames['Year'] == 2022]
              .groupby("Name")
              .sum()[["Count"]]
              .reset_index()
)
babynames_2022

<br>

---

### Question 3

Using the [pd.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) function described in the lecture, combine the `babynames_2022` table with the `elections_with_first_name` table you created earlier to form `presidential_candidates_and_name_popularity`. Your resulting `DataFrame` should contain all columns from both of the tables.

In [ ]:
presidential_candidates_and_name_popularity = ...
presidential_candidates_and_name_popularity

In [ ]:
grader.check("q3")

### **REVIEW:** `pandas.pivot_table`

Suppose we want to build a table showing the total number of babies born of each sex in each year. One way is to `groupby` using both columns of interest:

In [ ]:
babynames.groupby(["Year", "Sex"])[["Count"]].agg(sum).head(6)

While this does give us the information we're looking for, a more natural approach is to use pivot tables to represent our data in a more readable format.

In [ ]:
babynames_pivot = babynames.pivot_table(
    index = "Year",     # rows (turned into index)
    columns = "Sex",    # column values
    values = ["Count"], # field(s) to process in each group
    aggfunc = np.sum,   # group operation
)
babynames_pivot.head(6)

We can also include multiple values in our pivot tables 

In [ ]:
babynames_pivot = babynames.pivot_table(
    index = "Year",     # rows (turned into index)
    columns = "Sex",    # column values
    values = ["Count", "Name"],
    aggfunc = np.max,   # group operation
)
babynames_pivot.head(6)

<br>

---

### Question 4

Using `presidential_candidates_and_name_popularity`, create a table, `party_result_popular_vote_pivot`, whose index is the `Party` and whose columns are their `Result`. Each cell should contain the total number of popular votes received. `pandas`' `pivot_table` documentation is [here](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

You may notice that there are `NaN`s in your table from missing data. Replace the `NaN` values with 0. You may find `.pivot_table`'s `fill_value=` argument helpful. Or, you can use `pd.DataFrame.fillna` [(documentation here)](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html).

In [ ]:
...
party_result_popular_vote_pivot

In [ ]:
grader.check("q4")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Part 2

In Lab 3, we took you through the entire process of reading data from a file to perform some exploration of the data. Here, we again load the dataset that we will be using in Lab 4 along with some of the columns we had added in Lab 3. For any additional context regarding the dataset, we encourage you to revisit Lab 3.

In [ ]:
bus = pd.read_csv('data/bus.csv', encoding='ISO-8859-1').rename(columns={"business id column": "bid"})
bus['postal5'] = bus['postal_code'].str[:5]
ins = pd.read_csv('data/ins.csv')
ins['timestamp'] = pd.to_datetime(ins['date'], format='%m/%d/%Y %I:%M:%S %p')
ins['bid'] = ins['iid'].str.split("_", expand=True)[0].astype(int) 

# This code is essential for the autograder to function properly. Do not edit.
ins_test = ins

<br/>

---

## 🔎 Question 5: Inspecting the Inspections

### 🚀 Question 5a

Let's start by looking again at the first 5 rows of `ins` to see what we're working with.

In [ ]:
ins.head(5)

To better understand how the scores have been allocated, let's examine how the maximum score varies for each type of inspection. 

Create a `DataFrame` object `ins_score_by_type`, indexed by all the inspection types (e.g., New Construction, Routine - Unscheduled, etc.), with a single column named `max_score` containing the highest score received. Additionally, order `ins_score_by_type` by `max_score` in descending order. 

**Hint:** You may find the `rename` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)) to be useful! 

In [ ]:
ins_score_by_type = ...
ins_score_by_type

In [ ]:
grader.check("q5a")

<br/>

---

### 🚀 Question 5b


Given the variability of `ins['score']` observed in 1a, let's examine the inspection scores `ins['score']` further.

In [ ]:
ins['score'].value_counts().head()

There are a large number of inspections with a score of -1. These are probably missing values. Let's see what types of inspections have scores and which do not (score of -1). 

- First, define a new column `Missing Score` in `ins` where each row maps to the string `"Yes"` if the `score` for that business is -1 and `"No"` otherwise. 

- Then, use `groupby` to find the number of inspections for every combination of `type` and `Missing Score`. Store these values in a new column `Count`. 

- Finally, sort `ins_missing_score_group` by descending `Count`s. 
The result should be a `DataFrame` that looks like the one shown below.

**Hint**: You may find the `map` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html)) useful for defining `Missing Score`! 

<table border="1" class="dataframe">
  <caption>example dataframe</caption>
  <thead>
    <tr>
      <th scope="col"></th>
      <th scope="col"></th>
      <th scope="col">Count</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">type</th>
      <th scope="row">Missing Score</th>
      <th scope="row"></th>
    </tr>
    <tr>
    </tr>
    <tr>
      <th scope="row">Routine - Unscheduled</th>
      <th scope="row">No</th>
      <td>14031</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>

In [ ]:
ins['Missing Score'] = ...
ins_missing_score_group = ...


ins_missing_score_group

In [ ]:
grader.check("q5b")

<br/>

---

### 🚀 Question 5c


Using `groupby` to perform the analysis above gave us a `DataFrame` that wasn't the most readable at first glance. There are better ways to represent the information above that take advantage of the fact that we are looking at combinations of two variables. It's time to pivot (pun intended)!

Create a `DataFrame` that looks like the one below, and assign it to the variable `ins_missing_score_pivot`. 

You'll want to use the `pivot_table` method of the `DataFrame` class, which you can read about in the `pivot_table` [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html). 

- Once you create `ins_missing_score_pivot`, add another column titled `Percentage Missing`, which contains the proportion of missing scores within each `type`. 

- Then, sort `ins_missing_score_pivot` in ascending order of `Percentage Missing`. Reassign the sorted `DataFrame` back to `ins_missing_score_pivot`.

**Hint:** Consider what happens if no values correspond to a particular combination of `Missing Score` and `type`. Looking at the documentation for `pivot_table`, is there any function argument that allows you to specify what value to fill in?

If you've done everything right, you should observe that inspection scores appear only to be assigned to `Routine - Unscheduled` inspections and that `ins_missing_score_pivot` looks like the `DataFrame` below:


<table border="1" class="dataframe">
  <caption>example dataframe</caption>
  <thead>
    <tr>
      <th scope="col">Missing Score</th>
      <th scope="col">No</th>
      <th scope="col">Yes</th>
      <th scope="col">Percentage Missing</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">type</th>
      <th scope="row"></th>
      <th scope="row"></th>
      <th scope="row"></th>
    </tr>
    <tr>
      <th scope="row">Routine - Unscheduled</th>
      <td>14031</td>
      <td>46</td>
      <td>0.003268</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>


In [ ]:
ins_missing_score_pivot = ...

...

ins_missing_score_pivot

In [ ]:
grader.check("q5c")

Notice that inspection scores appear only to be assigned to `Routine - Unscheduled` inspections. Also, it is reasonable for inspection types such as `New Ownership` and `Complaint` to have no associated inspection scores, but you might be curious why there are no inspection scores for the `Reinspection/Followup` inspection type. Later in the HW, we will examine these `Reinspection/Followup` inspections.

<br/>

---

## 🚀 Question 6: Joining Data Across Tables

In this question, we will start to connect data across multiple tables. We will be using the `pd.merge` function. 

<br/>

--- 

### 🚀 Question 6a

Let's figure out which restaurants had the lowest scores. Before we proceed, filter out missing scores from `ins` so that negative scores don't influence our results. 

In [ ]:
ins = ins[ins["score"] > 0]

We'll start by creating a new `DataFrame` called `ins_named`. `ins_named` should be exactly the same as `ins`, except that it should have the name and address of every business, as determined by the `bus` `DataFrame`. 

**Hint**: Use the `DataFrame` method `merge` to join the `ins` `DataFrame` with the appropriate portion of the `bus` `DataFrame`. See the [documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html) for guidance on how to use `merge` function to combine two `DataFrame` objects. The first few rows of `ins_named` `DataFrame` are shown below:

<img src="images/2a.png" width="1080" alt="dataframe"/>

In [ ]:
...
ins_named.head()

In [ ]:
grader.check("q6a")

<br/>

--- 

### 🚀 Question 6b

Look at the 20 businesses in `ins_named` with the lowest scores. Order `ins_named` by each business's minimum score in ascending order. Use the business names in ascending order to break ties. The resulting `DataFrame` should look like the table below.

This one is pretty challenging! Don't forget to rename the `score` column. 

**Hint**: The `agg` function can accept a dictionary as an input. See the `agg` [documentation](https://pandas.pydata.org/pandas-docs/version/0.22/generated/pandas.core.groupby.DataFrameGroupBy.agg.html). Additionally, when thinking about what aggregation functions to use, ask yourself what value would be in the `name` column for each entry across the group? Can we select just one of these values to represent the whole group?

As usual, **YOU SHOULD NOT USE LOOPS OR LIST COMPREHENSIONS**. Try to break down the problem piece by piece instead, gradually chaining together different `pandas` functions. Feel free to use more than one line!

<table border="1" class="dataframe">
  <caption>example dataframe</caption>
  <thead>
    <tr>
      <th scope="col"></th>
      <th scope="col">name</th>
      <th scope="col">min score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th align="right" scope="row">bid</th>
      <th scope="row"></th>
      <th scope="row"></th>
    </tr>
    <tr>
      <th scope="row">86718</th>
      <td>Lollipot</td>
      <td>45</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>

In [ ]:
twenty_lowest_scoring = ... 

# DO NOT USE LIST COMPREHENSIONS OR LOOPS OF ANY KIND!!!

...

twenty_lowest_scoring

In [ ]:
grader.check("q6b")

<br/>

--- 
### 🚀 Question 6c
Let's do some more interesting analysis with our lowest score calculations. In the cell below, assign `worst_3_inspection_restaurants` to a two-column `DataFrame` with 15 rows.

- One column is the `name` of each business.

- The other column is a modified average inspection score of each business called `lowest 3 average`. 

    - To calculate `lowest 3 average`, find the average of each business's **three lowest inspection scores**. 

    - If a business has less than three inspection scores, take the average of all of its inspection scores (i.e., either one or two scores). 

- Finally, assign `worst_3_inspection_restaurants` to a `DataFrame` of the 15 rows with the lowest `lowest 3 average`, sorted by `lowest 3 average` ascending.  

`worst_3_inspection_restaurants` should look like the one below.

**Hint**: 6b’s advice also applies here! Furthermore, your answer to 6b may be helpful as a starting point. This question is intentionally left open-ended, so feel free to use any combination of `pandas` functions found online. Similarly to 6b, do not use loops or list comprehensions. Use as many lines as you see fit, so long as your final answer is saved to `worst_3_inspection_resturants`. For isolating each business's lowest three inspection scores, it may be useful to know that when `groupby` applies aggregating functions, it preserves the sort order of the inputted `DataFrame`. 


<table border="1" class="dataframe">
  <caption>example dataframe</caption>
  <thead>
    <tr>
      <th scope="col"></th>
      <th scope="col">name</th>
      <th scope="col">lowest 3 average</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th align="right" scope="row">bid</th>
      <th scope="row"></th>
      <th scope="row"></th>
    </tr>
    <tr>
      <th scope="row">84590</th>
      <td>Chaat Corner</td>
      <td>54.0</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>


In [ ]:
worst_3_inspection_restaurants = ...

...
worst_3_inspection_restaurants

In [ ]:
grader.check("q6c")

<br/>

---

## 🌮 Question 7: `pandas` Potpourri

In this question, we ask you to describe `pandas` operations and explain specific concepts using `ins_named`.

<!-- BEGIN QUESTION -->

<br/>

---

### 🌮 Question 7a

Consider the chained `pandas` statement below:

`q7a_df = ins_named[ins_named["name"].str.lower().str.contains("taco")].groupby("bid").filter(lambda sf: sf["score"].max() > 95).agg("count")`

We can decompose this statement into three parts:

```
temp1 = ins_named[ins_named["name"].str.lower().str.contains("taco")]
 
temp2 = temp1.groupby("bid").filter(lambda sf: sf["score"].max() > 95)
 
q3a_df = temp2.agg("count")
```

For each line of code above, write one sentence describing what the line of code accomplishes. Feel free to create a cell to see what each line does. In total, you'll write three sentences.

Finally, write an example homework question whose answer is `q7a_df`. 

- This example homework question should only be one sentence.

**Note: While the first part of this question will be graded for correctness, the second part is a bit more open-ended. Answers that demonstrate correct understanding will receive full credit.** 

An example answer will look like the following: "`temp1` creates a ... `temp2` transforms `temp1` by ... Finally, `q7a_df` results in a `DataFrame` that ... A question that is answered by this chain of operations is ..."

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br/>

---

### 🌮 Question 7b

Consider `ins_named`, `temp1`, `temp2`, and `q7a_df` from the previous problem. What is the granularity of each `DataFrame`? Explain your answer in no more than four sentences.

**Note**: For more details on what the granularity of a `DataFrame` means, feel free to check the [course notes](https://ds100.org/course-notes/eda/eda.html#granularity)!

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

---

## 🚀 Question 8: Missing Inspections

With our inspection data, we are given the `type` of each inspection. These categories were lightly investigated in Question 5, centered on the number of missing scores within each `type`. Since the `timestamp` and `score` for each inspection are also provided, we can do a more interesting analysis relating the `score` and `timestamp` of specific types of inspections. 

Specifically, in Question 8, we are interested in the possible relationship between inspections of the `type` "Routine - Unscheduled" and "Reinspection/Followup" (the two most frequent inspection types in our dataset). We might guess that a follow-up ("Reinspection/Followup") inspection occurs more frequently when an initial ("Routine - Unscheduled") inspection receives a low score. To confirm this hunch, let’s investigate the rate of follow-up inspections for different initial scores. To simplify your analysis, we have provided a new `DataFrame` (`reinspections`). 

- `reinspections` contains every "Routine - Unscheduled" inspection, along with the relevant `bid` and `name` associated with the initial inspection. 
- `routine timestamp` indicates when the initial inspection occurred. 
- `routine score` is the score that the initial inspection received. 
- `day difference` is the number of days between the initial inspection and a follow-up inspection if done within one year. 
    
Some initial inspections did not have any follow-up inspections within one year. In these cases, `day difference` is assigned a filler value of -1.

Run the cell below to load in `reinspections`.

In [ ]:
reinspections = pd.read_csv('data/reinspections.csv')
reinspections

<br/>

--- 
### 🚀 Question 8a
First, create a new `Boolean` column `recent reinspection?` that indicates whether a follow-up inspection occurred within 62 days inclusive (~2 months) of an initial inspection. 

In [ ]:
reinspections['recent reinspection?'] = ...

reinspections

In [ ]:
grader.check("q8a")

<br/>

--- 
### 🚀 Question 8b
To simplify our analysis, let’s assign `routine score`s to buckets. Buckets are similar to the bins of a histogram. Each bucket contains all scores that fall in a particular range.

Below we have defined the function `bucketify`. Use `bucketify` to create a new column in `reinspections` called `score buckets` that **maps** the score of an initial inspection to one of these predefined buckets.

**Hint:** You may find the `map` [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html) useful. Alternatively, see [this demonstration](https://ds100.org/course-notes/pandas_3/pandas_3.html#approach-3-sorting-using-the-map-function) in the course notes for an example use case.

In [ ]:
def bucketify(score):
    if score < 65: 
        return '0 - 65'
    elif score < 70:
        return '65 - 69'
    elif score < 75:
        return '70 - 74'
    elif score < 80:
        return '75 - 79'
    elif score < 85:
        return '80 - 74'
    elif score < 90:
        return '85 - 89'
    elif score < 95:
        return '90 - 94'
    else:
        return '95 - 100'
        
reinspections['score buckets'] = ...

reinspections

In [ ]:
grader.check("q8b")

<br/>

--- 
### 🚀 Question 8c
Before we complete our analysis, remove all rows whose `score buckets` contain less than 125 rows. Assign `reinspection_filtered` to this new `DataFrame`. 

In [ ]:
reinspections_filtered = ...

reinspections_filtered

In [ ]:
grader.check("q8c")

<br/>

--- 
### 🚀 Question 8d

To conclude our analysis, use `resinpsections_filtered` to generate a `DataFrame` with the **proportion** of initial inspections within each bucket that were reinspected within 62 days, along with the total **count** of initial inspections included in each bucket. Sort this `DataFrame` by ascending counts. Assign this new `DataFrame` to `reinspection_proportions`.

`reinspection_proportions` should look like the `DataFrame` below.

<table border="1" class="dataframe">
  <caption>example dataframe</caption>
  <thead>
    <tr>
      <th scope="col"></th>
      <th scope="col">recent reinspection?</th>
      <th scope="col"></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row"></th>
      <th scope="row">proportion</th>
      <th scope="row">count</th>
    </tr>
    <tr>
      <th scope="row">score buckets</th>
      <th scope="row"></th>
      <th scope="row"></th>
    </tr>
    <tr>
      <th scope="row">70 - 74</th>
      <td>0.407821</td>
      <td>358</td>
    </tr>
    <tr>
      <th scope="row">...</th>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>

In [ ]:
reinspection_proportions = ...

reinspection_proportions

In [ ]:
grader.check("q8d")

<!-- BEGIN QUESTION -->

<br/>

--- 
### 🚀 Question 4e

Do you notice any trends? Are your results consistent with your prior knowledge about restaurants that receive high or low health inspection scores? Answer in the cell below.

**This question is graded on effort, there is no one "correct" answer.**

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Summary of Inspections Data

We have done a lot in this lab! 
 
- Broke down the inspection scores in detail using `groupby` and `pivot_table`.
- Joined the business and inspection data and identified restaurants with the worst ratings.
- Took a deep dive into understanding any trends between an inspection score and reinspection frequency.

Over the course of this 2-part lab, we hope you have become more familiar with `pandas` - in terms of identifying when to use particular functions, how they work, when they can support EDA - as well as with EDA and Data Cleaning, as part of the broader Data Science Lifecycle. These tools will serve you well as a data scientist!

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Lab 4! ##

### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Once you submit this file to the Lab 4 assignment on Gradescope, Gradescope will automatically submit a PDF file with your written answers to the Lab 4 Written assignment.

**Important**: Please check that your written responses were generated and submitted correctly to the Lab 4 Written Assignment.

**You are responsible for ensuring your submission follows our requirements and that the PDF for Lab 4 written answers was generated/submitted correctly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out prior to the deadline. 

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)